# Data Preparation

## Objective

Prepare a candidate feature set for the enrollment-time student-outcome prediction task.

This notebook will:

1. Load the original dataset.
2. identify and exclude first- and second-semester features.
3. Separate the candidate input features from the target.
4. Verify the resulting dimensions and target distribution.

This is a preliminary preparation stage. Features whose availability at enrollment is uncertain will remain subject to review before the final model is trained.

The raw dataset will not be modified.

In [5]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent
data_path = project_root / "data" / "raw" / "data.csv"

df = pd.read_csv(data_path, sep=";")
df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
print("Target column exists:", "Target" in df.columns)

Dataset shape: (4424, 37)
Target column exists: True


## Excluding Post-Enrollment Features

The intended prediction point is enrollment. The first- and second-semester features describe academic activity or performance that occurs after that point.

These features are excluded from the candidate enrollment-time input set to avoid temporal data leakage.

In [6]:
semester_columns = [
    col for col in df.columns
    if "1st sem" in col.lower()
    or "2nd sem" in col.lower()
]

print("Number of semester-related columns:", len(semester_columns))

for col in semester_columns:
    print("-", col)

Number of semester-related columns: 12
- Curricular units 1st sem (credited)
- Curricular units 1st sem (enrolled)
- Curricular units 1st sem (evaluations)
- Curricular units 1st sem (approved)
- Curricular units 1st sem (grade)
- Curricular units 1st sem (without evaluations)
- Curricular units 2nd sem (credited)
- Curricular units 2nd sem (enrolled)
- Curricular units 2nd sem (evaluations)
- Curricular units 2nd sem (approved)
- Curricular units 2nd sem (grade)
- Curricular units 2nd sem (without evaluations)


## Excluding Features with Uncertain Timing

In addition to the first- and second-semester features, three variables have been excluded from the initial strict enrollment-time experiment because their exact recording time has not been verified:

- `Debtor`
- `Tuition fees up to date`
- `Scholarship holder`

This is a conservative choice intended to reduce the risk of temporal data leakage. These features may be reconsidered in a separate experiment if their availability at enrollment can be established.

The original dataset is not modified.

In [9]:
uncertain_timing_columns = [
    "Debtor",
    "Tuition fees up to date",
    "Scholarship holder",
]

excluded_columns = semester_columns + uncertain_timing_columns

print("Semester-related columns:", len(semester_columns))
print("Features excluded for uncertain timing:", len(uncertain_timing_columns))
print("Total excluded input features:", len(excluded_columns))

for col in uncertain_timing_columns:
    assert col in df.columns, f"Expected column not found: {col}"

Semester-related columns: 12
Features excluded for uncertain timing: 3
Total excluded input features: 15


## Creating the Initial Enrollment-Time Feature Set

We will create `X` from the original dataset by excluding the target, all first- and second-semester features, and the three variables whose timing has not yet been verified.

The target `y` remains `Target`, which contains the final student outcome.

We will preserve the original DataFrame so that later experiments can use a different, explicitly documented feature set.

In [10]:
candidate_feature_columns = [
    col
    for col in df.columns
    if col != "Target" and col not in excluded_columns
]

X = df[candidate_feature_columns].copy()
y = df["Target"].copy()

print("Original dataset shape:", df.shape)
print("Number of selected features:", len(candidate_feature_columns))
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nSelected features:")
for col in candidate_feature_columns:
    print("-", col)

print("\nTarget distribution:")
print(y.value_counts())

Original dataset shape: (4424, 37)
Number of selected features: 21
X shape: (4424, 21)
y shape: (4424,)

Selected features:
- Marital status
- Application mode
- Application order
- Course
- Daytime/evening attendance
- Previous qualification
- Previous qualification (grade)
- Nacionality
- Mother's qualification
- Father's qualification
- Mother's occupation
- Father's occupation
- Admission grade
- Displaced
- Educational special needs
- Gender
- Age at enrollment
- International
- Unemployment rate
- Inflation rate
- GDP

Target distribution:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


In [11]:
assert "Target" not in X.columns

assert not any(
    "1st sem" in col.lower() or "2nd sem" in col.lower()
    for col in X.columns
)

assert not any(
    col in X.columns
    for col in uncertain_timing_columns
)

assert X.shape[1] == 21
assert len(X) == len(y)
assert df.shape == (4424, 37)

print("All feature-selection checks passed.")
print("Input features:", X.shape[1])
print("Input rows:", X.shape[0])
print("Target rows:", len(y))

All feature-selection checks passed.
Input features: 21
Input rows: 4424
Target rows: 4424
